# Reproduction: training model

This jupyter notebook illustrates how to reproduce the model training (and evaluation) process of the linear regression model and the ridge regression model. Besides, all code parts used in this notebook can be found in the project repo.

# Procedure
1. Preparation
2. Linear Regression Model (cf. reg_ols.pkl)
3. Ridge Regression Model (cf. ridge.pkl)

The training data we used is from BPI Challenge 2013, which can be either found in the training_data directory in this project, or be downloaded from https://www.processmining.org/event-data.html, aka. Incident management log (BPI Challenge 2013).

# Step 1: Preparation

Before we step into the linear regression model and the ridge regression model, we first established a baseline model which will provide a frame for comparison for those two models. The following cell is analogous to pipeline_demo.py in the sub-directory remaining_time.

We first preprocess the data and calculate the mean remaining time.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

code_folder = os.path.join(os.getcwd(), 'remaining_time')
if code_folder not in sys.path:
    sys.path.append(code_folder)

from remaining_time.pipeline_helper import numeric_split, preprocess_data
from remaining_time.baselines import mean_predictor
from remaining_time.model_evaluation import myScore
from remaining_time.pipeline_helper import get_log_with_length_index, get_variants
from remaining_time.checker import df_type_check

train_log, val_log, test_log = preprocess_data()

x_train, y_train = numeric_split(train_log, "remaining_time")
x_test, y_test = numeric_split(test_log, "remaining_time")

mean_remaining_time = mean_predictor(y_train)

--- Starting pipeline for remaining time ---
Loading log...


c:\Users\20446\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
c:\Users\20446\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 7554/7554 [00:04<00:00, 1523.37it/s]


Log loaded successfully.
Validating columns...
Columns validated successfully.
Sorting cases by timestamp...
Cases sorted successfully.
Filtering for completed cases ending with ['Completed']...
Computing remaining time for each prefix...
Remaining time for each prefix has been computed successfully.
Filtering cases with less than 2 events...
Cases have been filtered successfully.
Extracting static case attributes for ['impact', 'product', 'organization involved']...
Extracting aggregated dynamic features...
Extracting temporal features...
Applying one-hot encoding to ['concept:name', 'static_impact', 'static_product', 'static_organization involved']...
Performing time-based training/validation/testing split...
Data splitting has been completed successfully. The data has been split into: 5282 training, 1131 validation and 1133 test cases. Total: 7546 cases.
Scaling numerical features using StandardScaler...
 --- Successfully scaled numeric features and saved scaler.pkl --- 
--- Running

Then we can evaluate and store the evaluation results in model_metrics.csv and model_scores.csv.

In [2]:
# analogous to baselines.save_baseline, but instead save the .csv outputs in the notebooks directory
df_type_check(test_log)
data_model_scores = {
    'name':['baseline'],
    'best':['Y'],
    'abs_super':[0],                    # according to the definition of abs_super, identical inputs always result in 0
    'details':['This model predicts every input simply as the mean value of time of the training data. ']
}
df1 = pd.DataFrame(data_model_scores)
df1.to_csv("notebooks/model_scores.csv", index=False)
print(" --- model_scores.csv successfully created --- ")

df2 = pd.DataFrame(columns=['name', 'prefix_length', 'MAE', 'RMSE', 'MedAE', 'R2'])
res_list = []
for i in range(len(get_variants(test_log))):
    y_test = get_log_with_length_index(test_log, i).iloc[:,-1]
    y_pred = np.full(y_test.shape, mean_remaining_time)
    score = myScore(pd.DataFrame(y_test), pd.DataFrame(y_pred))
    row = {
        'name':'baseline',
        'prefix_length': get_variants(test_log)[i],
        'MAE':score[0],
        'RMSE':score[1],
        'MedAE':score[2],
        'R2':score[3]
    }
    res_list.append(row)
df2 = pd.DataFrame(res_list)
df2.to_csv("notebooks/model_metrics.csv", index=False)
print(" --- model_metrics.csv successfully created --- ")

 --- model_scores.csv successfully created --- 
 --- model_metrics.csv successfully created --- 


Hence, you can directly find the created model_metrics.csv and model_scores.csv in the notebooks directory. All cells in the .csv files should be findable in the according .csv files in the main directory, except that the baseline still holds a 'Y' in the best column, because there's no model else which exists now. Ditto, the same problem will also trigger in the linear regression model reproduction.

# Part 2: Linear Regression Model

The linear regression model we used is the simplest ordinary least square model, therefore, it's named as reg_ols.pkl.

Again, we start with preprocessing data.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from remaining_time.pipeline_helper import preprocess_data, numeric_split
from remaining_time.score_management import add_list_of_lines, update_and_set
from remaining_time.model_evaluation import abs_super, rel_super, loose_compare

print( " --- start preprocessing data --- ")
train_data, val_data, test_data = preprocess_data()
x_train, y_train = numeric_split(train_data, "remaining_time")
print(" --- data prepared --- ")

 --- start preprocessing data --- 
--- Starting pipeline for remaining time ---
Loading log...


parsing log, completed traces :: 100%|██████████| 7554/7554 [00:04<00:00, 1705.39it/s]


Log loaded successfully.
Validating columns...
Columns validated successfully.
Sorting cases by timestamp...
Cases sorted successfully.
Filtering for completed cases ending with ['Completed']...
Computing remaining time for each prefix...
Remaining time for each prefix has been computed successfully.
Filtering cases with less than 2 events...
Cases have been filtered successfully.
Extracting static case attributes for ['impact', 'product', 'organization involved']...
Extracting aggregated dynamic features...
Extracting temporal features...
Applying one-hot encoding to ['concept:name', 'static_impact', 'static_product', 'static_organization involved']...
Performing time-based training/validation/testing split...
Data splitting has been completed successfully. The data has been split into: 5282 training, 1131 validation and 1133 test cases. Total: 7546 cases.
Scaling numerical features using StandardScaler...
 --- Successfully scaled numeric features and saved scaler.pkl --- 
 --- data p

Then we can train the model using fit() method on the training data.

In [4]:
print(" --- start training linear regression model --- ")
reg_ols = LinearRegression()
pipe = make_pipeline(reg_ols)
pipe.fit(x_train, y_train)

coef = reg_ols.coef_
coef_string = np.array2string(coef, precision=4, separator=', ')
incp = reg_ols.intercept_
result_string = f"coefficient: {coef_string}, intercept: {incp}"
print(" --- training completed --- ")
print(result_string)

 --- start training linear regression model --- 
 --- training completed --- 
coefficient: [[ 2.2766e+01, -1.7509e+02,  3.7599e+01,  8.9042e+01,  7.4889e+01,
  -6.6387e+01, -2.6860e+01, -3.2297e+00, -8.2765e+01,  1.5310e+01,
  -9.4499e+00, -5.2866e+01, -1.2200e+01,  9.9106e+00,  2.2379e+01,
   1.1657e+01,  2.4474e+01, -1.2145e+00,  4.1327e+01,  2.7182e+01,
   4.2665e+01,  3.5018e+00,  3.8473e+01, -8.1824e+01, -5.7252e+01,
  -4.8858e+01,  3.4674e+01, -1.2577e+00, -8.4348e+00, -1.1005e+01,
   6.5059e+00,  3.6843e+01,  1.2396e+01, -2.7626e+01,  3.3194e+01,
  -1.7972e+02,  1.0999e+02,  3.6533e+01, -8.2652e+01,  1.0019e+02,
  -8.6208e+01,  6.8674e+01, -1.1365e+02, -1.0587e+02, -1.4030e+02,
  -9.0949e-13, -3.2383e+02, -8.6439e+01, -1.6551e+02, -1.7053e-12,
   1.4113e+02,  3.9857e+01,  8.2423e-13, -2.3516e+02, -3.5588e+02,
  -1.1937e-12,  4.7748e-12,  5.3544e+02,  5.6838e+01,  3.1076e+03,
  -2.8081e+02, -5.0175e+01, -1.1640e+02, -1.2926e+02, -1.0645e+02,
  -1.6343e+02,  1.7053e-13,  3.7892e+0

finally, we can evaluate the model based on the testing data. However, since the trained models are equivalent in our project, if they share the same model_metrics and model_scores, we omit the model saving part.

It's remarkable that we demonstrate the model evaluation with a function entitled repro_evaluate_model, which is same as the evaluate_model function in model_evaluation.py. The only exception is that the paths of .csv files are changed because we stored the newly created .csv files in the notebooks directory now.

In [5]:
def repro_evaluate_model(model, model_name, test_data, result_string):
    df_type_check(test_data)
    print(" --- start evaluating model --- ")
    df = pd.read_csv("notebooks/model_metrics.csv")
    new_rows = []
    variants = get_variants(test_data)
    
    for i in range(len(variants)):
        variant = variants[i]
        log_data = get_log_with_length_index(test_data, i)
        
        x_test, y_test = numeric_split(log_data, "remaining_time")
        y_pred = model.predict(x_test)
        score = myScore(y_test, pd.DataFrame(y_pred))
        
        new_rows.append({
            "name": model_name,
            "prefix_length": variant,
            "MAE": score[0],
            "RMSE": score[1],
            "MedAE": score[2],
            "R2": score[3]
        })
    df = add_list_of_lines(df, new_rows)
    df.to_csv("notebooks/model_metrics.csv", index=False)
    print(" --- model metrics successfully stored --- ")

    df1 = pd.read_csv("notebooks/model_metrics.csv")
    df2 = pd.read_csv("notebooks/model_scores.csv")
    abs_su = abs_super(df1, model_name, test_data)                    
    rel_su = rel_super(df1, df2, model_name, test_data)                     
    update_info = loose_compare(model_name, rel_su)             

    print(" --- loose compare done --- ")

    df2 = update_and_set(df2, update_info, abs_su, result_string)
    df2.to_csv("notebooks/model_scores.csv", index=False)
    print(" --- model score successfully stored and updated --- ")

repro_evaluate_model(reg_ols, "ols", test_data, result_string)
print(" --- model evaluation done --- ")

 --- start evaluating model --- 
 --- model metrics successfully stored --- 
 --- loose compare done --- 
 --- model score successfully stored and updated --- 
 --- model evaluation done --- 


Since the linear regression model should perform better than the baseline model, you can see form model_scores.csv that now the ols holds the place of the current best model, i.e. a 'Y' in the 'best' column, while the baseline model is now not preferred.

# Part 3: Ridge Regression Model

The last model we want to illustrate is the ridge regression model, which also contains a hyperparameter alpha. Therefore, we decided to use grid search cross validation to find out the best alpha value to be set for this model. 

Since we used our self-defined score, we should first define the score explicitly here.

In [9]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import make_scorer
from remaining_time.model_evaluation import validScore

print(" --- start making score --- ")
def score_func(y_valid, pred):
    score_list = myScore(pd.DataFrame(y_valid), pd.DataFrame(pred))
    valid_score = validScore(score_list)
    return valid_score
score = make_scorer(score_func, greater_is_better=True)
print(" --- score established --- ")

 --- start making score --- 
 --- score established --- 


Let's preprocess data again. However, this time we also need the validation data for the cross validation.

In [10]:
print( " --- start preprocessing data --- ")
train_data, val_data, test_data = preprocess_data()
x_train, y_train = numeric_split(train_data, "remaining_time")
x_valid, y_valid = numeric_split(val_data, "remaining_time")

# combine the validation set and training set for cross validation using predefined split
x_combined = pd.concat([x_train, x_valid])
y_combined = pd.concat([y_train, y_valid])
train_indices = np.full(x_train.shape[0], -1)
val_indices = np.full(x_valid.shape[0], 0)
test_fold = np.concatenate([train_indices, val_indices])
split = PredefinedSplit(test_fold)
print(" --- data prepared --- ")

 --- start preprocessing data --- 
--- Starting pipeline for remaining time ---
Loading log...


parsing log, completed traces :: 100%|██████████| 7554/7554 [00:03<00:00, 2067.66it/s]


Log loaded successfully.
Validating columns...
Columns validated successfully.
Sorting cases by timestamp...
Cases sorted successfully.
Filtering for completed cases ending with ['Completed']...
Computing remaining time for each prefix...
Remaining time for each prefix has been computed successfully.
Filtering cases with less than 2 events...
Cases have been filtered successfully.
Extracting static case attributes for ['impact', 'product', 'organization involved']...
Extracting aggregated dynamic features...
Extracting temporal features...
Applying one-hot encoding to ['concept:name', 'static_impact', 'static_product', 'static_organization involved']...
Performing time-based training/validation/testing split...
Data splitting has been completed successfully. The data has been split into: 5282 training, 1131 validation and 1133 test cases. Total: 7546 cases.
Scaling numerical features using StandardScaler...
 --- Successfully scaled numeric features and saved scaler.pkl --- 
 --- data p

After that, we set up a grid search cross validation for training. The expected best alpha value should be 100.

In [11]:
print(" --- start grid search cross validation --- ")
model = Ridge()
param_grid = [{'alpha':[0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]}]
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=split,
    scoring=score
)

grid_search.fit(x_combined, y_combined)
best_alpha = grid_search.best_params_['alpha']
print(f"Best alpha found: {best_alpha}")
print(" --- grid search cross validation done --- ")

 --- start grid search cross validation --- 
Best alpha found: 100
 --- grid search cross validation done --- 


Different from ols, we have a hyperparameter found. Therefore, we need to use the validation set combined with the training set to retrain the model based on the best alpha found.

In [12]:
print(" --- start retraining --- ")
cur_model = Ridge(alpha=best_alpha)
cur_model.fit(x_combined, y_combined)

coef = cur_model.coef_
coef_string = np.array2string(coef, precision=4, separator=', ')
incp = cur_model.intercept_
result_string = f"alpha: {best_alpha}, coefficient: {coef_string}, intercept: {incp}"
print(" --- retrain completed --- ")

 --- start retraining --- 
 --- retrain completed --- 


Finally, we evaluate the model based on testing data.

In [13]:
repro_evaluate_model(cur_model, "ridge", test_data, result_string)
print(" --- evaluation done --- ")

 --- start evaluating model --- 
 --- model metrics successfully stored --- 
 --- loose compare done --- 
 --- model score successfully stored and updated --- 
 --- evaluation done --- 


Now, it'clear to see in model_score.csv in the notebooks directory that the ridge regression model is the best.

# Part 4: Conclusion

The resulting model_metrics.csv and model_scores.csv should be identical to the ones having same name in the main directory now. 

Besides, it's remarkable that we don't have to perform strict comparison (cf. strict_compare() in model_evaluation.py), since we have one and only one model being the best model, which is the ridge regression model. 

However, the linear regression model we trained have a similar abs_super value, i.e. the super score against the baseline model, and thus it is possible that in other use cases outside the scope of our training data that ols can perform even better or nearly equally well as the ridge regression model.